In [1]:
import os
os.chdir(r'C:\Users\johnpaul\fraudguard-africa')
print(os.getcwd())

C:\Users\johnpaul\fraudguard-africa


In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score, precision_recall_curve, auc
import warnings
warnings.filterwarnings('ignore')

print("🚀 Loading FULL Dataset...")

df = pd.read_csv('data/PS_20174392719_1491204439457_log.csv')
print(f"Total rows: {len(df):,}")

# ====================== FEATURE ENGINEERING ======================
print("Feature Engineering in progress...")

df['balance_diff_orig'] = df['oldbalanceOrg'] - df['newbalanceOrig']
df['balance_diff_dest'] = df['oldbalanceDest'] - df['newbalanceDest']
df['amount_to_oldbalance_ratio'] = df['amount'] / (df['oldbalanceOrg'] + 1e-8)
df['amount_to_newbalance_ratio'] = df['amount'] / (df['newbalanceOrig'] + 1e-8)

df['hour'] = df['step'] % 24
df['is_night'] = df['hour'].isin([0,1,2,3,4,5,22,23]).astype(int)

df = pd.get_dummies(df, columns=['type'], prefix='type', drop_first=True)

orig_freq = df['nameOrig'].value_counts()
df['orig_transaction_freq'] = df['nameOrig'].map(orig_freq)

df['high_risk_transaction'] = ((df['type_CASH_OUT'] == 1) & (df['amount'] > 100000)).astype(int)

# ====================== PREPARE DATA ======================
drop_cols = ['nameOrig', 'nameDest', 'isFlaggedFraud', 'step']
feature_cols = [col for col in df.columns if col not in drop_cols + ['isFraud']]

X = df[feature_cols]
y = df['isFraud']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Calculate scale_pos_weight (important for imbalance)
scale_pos_weight = len(y_train[y_train==0]) / len(y_train[y_train==1])
print(f"scale_pos_weight: {scale_pos_weight:.2f}")

# ====================== FINAL MODEL ======================
print("Training XGBoost on Full Data (Memory Optimized)...")

final_model = xgb.XGBClassifier(
    n_estimators=200,
    max_depth=7,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
    n_jobs=-1,
    eval_metric='aucpr',
    scale_pos_weight=scale_pos_weight   # This replaces SMOTE
)

final_model.fit(X_train, y_train)

# ====================== EVALUATION ======================
print("\n=== FINAL MODEL EVALUATION ===")
y_pred = final_model.predict(X_test)
y_pred_proba = final_model.predict_proba(X_test)[:, 1]

print(classification_report(y_test, y_pred, digits=4))
print(f"ROC-AUC : {roc_auc_score(y_test, y_pred_proba):.4f}")

precision, recall, _ = precision_recall_curve(y_test, y_pred_proba)
print(f"PR-AUC  : {auc(recall, precision):.4f}")

# ====================== SAVE ======================
os.makedirs('models', exist_ok=True)
joblib.dump(final_model, 'models/fraudguard_xgboost_full.pkl')
joblib.dump(feature_cols, 'models/feature_cols.pkl')

print("\n✅ Full Model Trained & Saved Successfully!")

🚀 Loading FULL Dataset...
Total rows: 6,362,620
Feature Engineering in progress...
scale_pos_weight: 773.75
Training XGBoost on Full Data (Memory Optimized)...

=== FINAL MODEL EVALUATION ===
              precision    recall  f1-score   support

           0     1.0000    0.9998    0.9999   1270881
           1     0.8418    0.9939    0.9115      1643

    accuracy                         0.9998   1272524
   macro avg     0.9209    0.9968    0.9557   1272524
weighted avg     0.9998    0.9998    0.9998   1272524

ROC-AUC : 0.9989
PR-AUC  : 0.9869

✅ Full Model Trained & Saved Successfully!
